https://github.com/UKPLab/sentence-transformers/blob/68dfbe643d51f1890e410b6783ca5343620db4fc/sentence_transformers/trainer.py#L620

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# Load a dataset (for example, IMDb for sentiment analysis)
imdb_dataset = load_dataset("imdb")
imdb_train_dataset = imdb_dataset['train'].shuffle().select(range(1000))  # Small subset for quick training
imdb_eval_dataset = imdb_dataset['test'].shuffle().select(range(500))

# Load a pre-trained model and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize the data
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

train_dataset = imdb_train_dataset.map(tokenize, batched=True)
eval_dataset = imdb_eval_dataset.map(tokenize, batched=True)


# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to = 'tensorboard'
)

#print(training_args.report_to)
#training_args.report_to = None

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('Ready for training!')
# Train the model
trainer.train()